# Part 5: Drive Diagnostics and Experiments

本 notebook 是 Part 1-4 完成后的后续实验入口。默认从 Google Drive 中已经保存的 `packs/` 归档恢复数据、推理结果和评估表，不重新运行 Part 1-4 主线。

适用任务：结果诊断、错误切片、后处理、自一致性推理评估、GRPO reward 分析，以及后续小规模消融实验。

## 固定前提

- `notebooks/part1.ipynb` 到 `notebooks/part4.ipynb` 已经运行完成。
- Drive 中存在 `part3_results_only_*.tar.gz` 和 `part4_results_only_*.tar.gz`。
- 后续实验优先读取保存结果和中间数据，新的训练或推理实验应另开 cell 或另建 notebook，不改 Part 1-4 的阶段结构。

## Cell 0.1 — 挂载 Drive 并定义路径

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import math
import tarfile
import pickle

import numpy as np
import pandas as pd

RUNTIME    = Path("/content/tsad_runtime")
DRIVE_ROOT = Path("/content/drive/MyDrive/tsad_anomaly")

RT_CODE    = RUNTIME / "code"
RT_SFT     = RUNTIME / "sft"
RT_CKPT    = RUNTIME / "checkpoints"
RT_RESULTS = RUNTIME / "results"

DRV_PACK   = DRIVE_ROOT / "packs"
DRV_SFT    = DRIVE_ROOT / "sft"
DRV_CKPT   = DRIVE_ROOT / "checkpoints"
DRV_RESULTS= DRIVE_ROOT / "results"
ANOMLLM    = RT_CODE / "AnomLLM"

SUBSETS = ["flat-trend", "range", "point", "freq"]

for p in [RUNTIME, RT_CODE, RT_SFT, RT_CKPT, RT_RESULTS,
          DRV_PACK, DRV_SFT, DRV_CKPT, DRV_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

print("RUNTIME:", RUNTIME)
print("DRV_PACK:", DRV_PACK)

## Cell 0.2 — 从 Drive 恢复已保存结果

先恢复 Part 3 的评估数据和 SFT 结果，再恢复 Part 4 的 GRPO 结果。这样可以得到同一 eval split 上的 zero-shot、SFT、GRPO 和 Isolation Forest 输出。

In [ ]:
def latest_archive(pattern: str, required: bool = True):
    matches = sorted(DRV_PACK.glob(pattern))
    if not matches:
        if required:
            raise FileNotFoundError(f"未找到 {DRV_PACK / pattern}")
        return None
    return matches[-1]


def extract_archive(path: Path):
    print("恢复归档:", path)
    with tarfile.open(path, "r:gz") as tar:
        tar.extractall(RUNTIME)

part3_pack = latest_archive("part3_results_only_*.tar.gz")
part4_pack = latest_archive("part4_results_only_*.tar.gz")

extract_archive(part3_pack)
extract_archive(part4_pack)

print("恢复完成")

## Cell 0.3 — 检查恢复结果

In [ ]:
required_paths = [
    RT_SFT / "sft_manifest.csv",
    RT_SFT / "eval.jsonl",
    RT_RESULTS / "baseline_compare.csv",
    RT_RESULTS / "sft_eval_metrics.csv",
    RT_RESULTS / "grpo_eval_metrics.csv",
]

for subset in SUBSETS:
    required_paths.extend([
        ANOMLLM / "data" / "synthetic" / subset / "eval" / "data.pkl",
        ANOMLLM / "results" / "synthetic" / subset / "qwen-local" / "0shot-vision.jsonl",
        ANOMLLM / "results" / "synthetic" / subset / "isolation-forest" / "0shot.jsonl",
        ANOMLLM / "results" / "synthetic" / subset / "sft-model" / "0shot-vision.jsonl",
        ANOMLLM / "results" / "synthetic" / subset / "grpo-model" / "0shot-vision.jsonl",
    ])

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("缺少恢复文件:
" + "
".join(missing))

print("关键文件齐全:", len(required_paths))
print("SFT metrics")
display(pd.read_csv(RT_RESULTS / "sft_eval_metrics.csv"))
print("GRPO metrics")
display(pd.read_csv(RT_RESULTS / "grpo_eval_metrics.csv"))

## Cell 1.1 — 读取 eval split 和预测输出

In [ ]:
def read_jsonl(path: Path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def coerce_intervals(value):
    if value is None:
        return None
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None
    return None


def parse_response(value):
    if isinstance(value, list):
        return value
    if value is None:
        return None
    if not isinstance(value, str):
        value = json.dumps(value, ensure_ascii=False)
    start = value.find("[")
    end = value.rfind("]")
    if start < 0 or end < start:
        return None
    try:
        parsed = json.loads(value[start:end + 1])
    except Exception:
        return None
    return parsed if isinstance(parsed, list) else None


def valid_intervals(intervals):
    if not isinstance(intervals, list):
        return False
    for item in intervals:
        if not isinstance(item, dict):
            return False
        if "start" not in item or "end" not in item:
            return False
        try:
            s = float(item["start"])
            e = float(item["end"])
        except Exception:
            return False
        if not s < e:
            return False
    return True


def interval_to_vector(intervals, length=1000):
    vec = np.zeros(length, dtype=np.int8)
    for item in intervals or []:
        try:
            s = int(round(float(item["start"])))
            e = int(round(float(item["end"])))
        except Exception:
            continue
        s = max(0, min(length, s))
        e = max(0, min(length, e))
        if s < e:
            vec[s:e] = 1
    return vec


def f1_score_intervals(pred, gt, length=1000):
    pred_vec = interval_to_vector(pred, length)
    gt_vec = interval_to_vector(gt, length)
    tp = int(((pred_vec == 1) & (gt_vec == 1)).sum())
    fp = int(((pred_vec == 1) & (gt_vec == 0)).sum())
    fn = int(((pred_vec == 0) & (gt_vec == 1)).sum())
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0
    if tp == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)


def gt_stats(intervals):
    intervals = intervals or []
    total = 0
    for item in intervals:
        try:
            total += max(0, int(item["end"]) - int(item["start"]))
        except Exception:
            pass
    if total == 0:
        bucket = "normal"
    elif total <= 30:
        bucket = "short"
    elif total <= 100:
        bucket = "medium"
    else:
        bucket = "long"
    return len(intervals), total, bucket


eval_rows = read_jsonl(RT_SFT / "eval.jsonl")
eval_df = pd.DataFrame(eval_rows)
eval_df["intervals_obj"] = eval_df["intervals"].apply(coerce_intervals)
print("eval rows:", len(eval_df))
display(eval_df.head())

## Cell 1.2 — 生成 parse 与 F1 诊断表

In [ ]:
METHODS = [
    ("isolation-forest", "isolation-forest", "0shot.jsonl"),
    ("qwen-local-0shot", "qwen-local", "0shot-vision.jsonl"),
    ("sft-0shot", "sft-model", "0shot-vision.jsonl"),
    ("grpo-0shot", "grpo-model", "0shot-vision.jsonl"),
]

loaded_results = {}
for subset in SUBSETS:
    for method, model_dir, filename in METHODS:
        path = ANOMLLM / "results" / "synthetic" / subset / model_dir / filename
        loaded_results[(subset, method)] = read_jsonl(path)

records = []
for _, row in eval_df.iterrows():
    subset = row["subset"]
    pkl_idx = int(row["pkl_idx"])
    gt = row["intervals_obj"] or []
    gt_n, gt_total_len, gt_len_bucket = gt_stats(gt)

    for method, _, _ in METHODS:
        result_rows = loaded_results[(subset, method)]
        result = result_rows[pkl_idx]
        parsed = parse_response(result.get("response"))
        parse_success = parsed is not None
        is_valid = valid_intervals(parsed)
        pred = parsed if is_valid else []
        records.append({
            "sample_id": row.get("sample_id"),
            "subset": subset,
            "pkl_idx": pkl_idx,
            "method": method,
            "label": row.get("label"),
            "anomaly_type": row.get("anomaly_type"),
            "gt_n_intervals": gt_n,
            "gt_total_len": gt_total_len,
            "gt_len_bucket": gt_len_bucket,
            "parse_success": parse_success,
            "valid_intervals": is_valid,
            "empty_pred": len(pred) == 0,
            "pred_n_intervals": len(pred),
            "f1": f1_score_intervals(pred, gt),
        })

diag = pd.DataFrame(records)
parse_summary = (
    diag.groupby("method")
    .agg(
        samples=("sample_id", "count"),
        parse_success_rate=("parse_success", "mean"),
        valid_rate=("valid_intervals", "mean"),
        empty_pred_rate=("empty_pred", "mean"),
        f1_mean=("f1", "mean"),
    )
    .reset_index()
)

by_subset = (
    diag.groupby(["method", "subset"])
    .agg(
        samples=("sample_id", "count"),
        parse_success_rate=("parse_success", "mean"),
        valid_rate=("valid_intervals", "mean"),
        empty_pred_rate=("empty_pred", "mean"),
        f1_mean=("f1", "mean"),
    )
    .reset_index()
)

by_interval = (
    diag.groupby(["method", "gt_len_bucket"])
    .agg(
        samples=("sample_id", "count"),
        parse_success_rate=("parse_success", "mean"),
        valid_rate=("valid_intervals", "mean"),
        f1_mean=("f1", "mean"),
    )
    .reset_index()
)

parse_summary.to_csv(RT_RESULTS / "diagnostics_parse.csv", index=False)
by_subset.to_csv(RT_RESULTS / "diagnostics_by_subset.csv", index=False)
by_interval.to_csv(RT_RESULTS / "diagnostics_by_interval.csv", index=False)
diag.to_csv(RT_RESULTS / "diagnostics_detail.csv", index=False)

print("wrote:")
print(RT_RESULTS / "diagnostics_parse.csv")
print(RT_RESULTS / "diagnostics_by_subset.csv")
print(RT_RESULTS / "diagnostics_by_interval.csv")
print(RT_RESULTS / "diagnostics_detail.csv")

display(parse_summary)
display(by_subset)
display(by_interval)

## Cell 1.3 — 只在双方 parse 成功样本上比较检测能力

这个表用于拆分“格式修复收益”和“检测能力收益”。

In [ ]:
wide = diag.pivot_table(
    index=["sample_id", "subset", "pkl_idx", "gt_len_bucket"],
    columns="method",
    values=["f1", "valid_intervals"],
    aggfunc="first",
).reset_index()

pairs = []
for other in ["sft-0shot", "grpo-0shot"]:
    base_ok = wide[("valid_intervals", "qwen-local-0shot")].fillna(False)
    other_ok = wide[("valid_intervals", other)].fillna(False)
    mask = base_ok & other_ok
    pairs.append({
        "comparison": f"qwen-local-0shot vs {other}",
        "samples_both_valid": int(mask.sum()),
        "qwen_f1_on_both": float(wide.loc[mask, ("f1", "qwen-local-0shot")].mean()),
        f"{other}_f1_on_both": float(wide.loc[mask, ("f1", other)].mean()),
    })

pair_df = pd.DataFrame(pairs)
pair_df.to_csv(RT_RESULTS / "diagnostics_valid_pair_compare.csv", index=False)
display(pair_df)

## Cell 1.4 — GRPO reward 诊断

In [ ]:
log_path = RT_RESULTS / "grpo_train_log_history.json"
if log_path.exists():
    logs = json.loads(log_path.read_text())
    reward_df = pd.DataFrame(logs)
    reward_cols = [c for c in reward_df.columns if "reward" in c.lower() or "loss" in c.lower()]
    keep_cols = [c for c in ["step", "epoch"] if c in reward_df.columns] + reward_cols
    reward_diag = reward_df[keep_cols].copy() if keep_cols else reward_df.copy()
    reward_diag.to_csv(RT_RESULTS / "grpo_reward_diagnostics.csv", index=False)
    display(reward_diag.tail(20))
else:
    print("未找到 GRPO 训练日志:", log_path)
    print("若要分析 reward 分布，请确认 part4_results_only 归档包含 grpo_train_log_history.json。")

## Cell 2.1 — 后处理实验占位

在这里追加区间裁剪、短区间过滤、相邻区间合并、自一致性投票等实验。每个实验需要写出新的 CSV，并与 `diagnostics_parse.csv` 中的当前基线比较。

In [ ]:
# 示例：复制 diag 后添加自己的 postprocess 结果。
# post_diag = diag.copy()
# post_diag["method"] = post_diag["method"] + "+postprocess"
# post_diag.to_csv(RT_RESULTS / "diagnostics_postprocess_candidate.csv", index=False)

print("后处理实验从这里开始。")